## Objective

- Discretize continuous state space into (inventory_bucket, days_bucket) pairs
- Initialize Q-table with zeros for all state-action combinations
- Build shared evaluation framework (run_episodes helper)

## Results

- Q-table shape confirmed: (10, 10, 10) — inventory x days x actions
- Discretization function tested against sample state values
- run_episodes() validated against RandomAgent on placeholder env

## Conclusion

State discretization and evaluation infrastructure are ready for
Q-Learning training in the coming days.

# Week 2 — Q-Learning Setup

## 1. Imports

In [ ]:
import sys
sys.path.append('../src')
from q_learning_agent import QLearningAgent
from eval_utils import run_episodes
import gymnasium as gym

print("Imports successful")

## 2. Initialize Q-Learning Agent

In [ ]:
agent = QLearningAgent()
print("Q-table shape:", agent.q_table.shape)
print("Sample discretization (inv=50, days=15):",
      agent.discretize_state(50, 15))

## 3. Test Evaluation Framework

In [ ]:
from random_agent import RandomAgent

test_env = gym.make("CartPole-v1")
random_agent = RandomAgent(test_env.action_space)

results = run_episodes(random_agent, test_env, n_episodes=50)
print("Mean revenue:", results["mean_revenue"])
print("Std revenue:", results["std_revenue"])
test_env.close()

## 4. Baseline Heuristic Comparison — Random vs Fixed vs Time-Based Discount

In [ ]:
from pricing_env import PricingEnv
from baseline_agents import FixedPriceAgent, TimeBasedDiscountAgent

env = PricingEnv()

random_agent = RandomAgent(env.action_space)
fixed_agent = FixedPriceAgent(price_level_index=5)
discount_agent = TimeBasedDiscountAgent(start_price_level=9)

results = {}
results["Random"] = run_episodes(random_agent, env, n_episodes=500)
results["Fixed"] = run_episodes(fixed_agent, env, n_episodes=500)
results["Discount"] = run_episodes(discount_agent, env, n_episodes=500)

for name, r in results.items():
    print(f"{name}: mean={r['mean_revenue']:.2f}, std={r['std_revenue']:.2f}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.boxplot([results[k]["all_rewards"] for k in results], tick_labels=list(results.keys()))
plt.title("Revenue Distribution — Baseline Heuristic Comparison")
plt.ylabel("Episodic Revenue")
plt.tight_layout()
plt.savefig('../outputs/baseline_comparison_boxplot.png', dpi=150)
plt.show()

## 5. Q-Learning Implementation Validation

Confirmed correct implementation:
- Bellman update: Q(s,a) += lr * (reward + gamma * max(Q(s')) - Q(s,a))
- Hyperparameters: learning_rate=0.1, discount_factor=0.95, epsilon=1.0 (decaying to 0.05)
- Epsilon-greedy action selection tested against real PricingEnv
- Reward accumulation verified via single-step test in q_learning_agent.py

## 6. Train Q-Learning Agent — 5,000 Episodes

In [ ]:
from pricing_env import PricingEnv
from q_learning_agent import QLearningAgent
import numpy as np
import matplotlib.pyplot as plt

env = PricingEnv()
q_agent = QLearningAgent()

n_training_episodes = 5000
training_rewards = []

for episode in range(n_training_episodes):
    obs, info = env.reset()
    total_reward = 0
    done = False

    while not done:
        action = q_agent.act(obs)
        next_obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

        q_agent.update(obs, action, reward, next_obs, done)
        obs = next_obs
        total_reward += reward

    q_agent.decay_epsilon()
    training_rewards.append(total_reward)

    if (episode + 1) % 500 == 0:
        avg_recent = np.mean(training_rewards[-500:])
        print(f"Episode {episode+1}/{n_training_episodes} | "
              f"Avg reward (last 500): {avg_recent:.2f} | Epsilon: {q_agent.epsilon:.3f}")

print("Training complete.")

## 7. Training Reward Curve

In [ ]:
# Smooth the reward curve using a rolling average for readability
window = 100
smoothed = np.convolve(training_rewards, np.ones(window)/window, mode='valid')

plt.figure(figsize=(8, 5))
plt.plot(smoothed)
plt.title(f"Q-Learning Training Reward Curve ({n_training_episodes} episodes, {window}-episode rolling avg)")
plt.xlabel("Episode")
plt.ylabel("Total Episodic Reward (smoothed)")
plt.tight_layout()
plt.savefig('../outputs/qlearning_training_curve.png', dpi=150)
plt.show()

## 8. Save Trained Q-Table

In [ ]:
import numpy as np

np.save('../outputs/trained_qtable.npy', q_agent.q_table)
print("Q-table saved to outputs/trained_qtable.npy")
print("Shape:", q_agent.q_table.shape)

## 9. Benchmark Trained Q-Learning Agent vs All Heuristic Baselines

In [ ]:
from baseline_agents import FixedPriceAgent, TimeBasedDiscountAgent
from random_agent import RandomAgent
from eval_utils import run_episodes

# Set epsilon to 0 for greedy evaluation (no exploration during benchmarking)
q_agent.epsilon = 0.0

agents = {
    "Random": RandomAgent(env.action_space),
    "Fixed": FixedPriceAgent(price_level_index=5),
    "Discount": TimeBasedDiscountAgent(start_price_level=9),
    "Q-Learning": q_agent,
}

benchmark_results = {}
for name, a in agents.items():
    benchmark_results[name] = run_episodes(a, env, n_episodes=500)

print(f"{'Agent':<12} {'Mean Revenue':>14} {'Std Dev':>10}")
for name, r in benchmark_results.items():
    print(f"{name:<12} {r['mean_revenue']:>14.2f} {r['std_revenue']:>10.2f}")

## 10. Sell-Through Rate Comparison

In [ ]:
def compute_sell_through(agent, env, n_episodes=500, max_inventory=100):
    sell_through_rates = []
    for _ in range(n_episodes):
        obs, info = env.reset()
        done = False
        while not done:
            action = agent.act(obs)
            obs, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
        final_inventory = obs[0]
        sell_through = (max_inventory - final_inventory) / max_inventory
        sell_through_rates.append(sell_through)
    return np.mean(sell_through_rates)

print(f"{'Agent':<12} {'Sell-Through Rate':>18}")
for name, a in agents.items():
    rate = compute_sell_through(a, env, n_episodes=200)
    print(f"{name:<12} {rate:>17.1%}")

## 11. Hyperparameter Tuning

In [ ]:
import itertools

learning_rates = [0.05, 0.1, 0.2]
discount_factors = [0.9, 0.95, 0.99]
epsilon_decays = [0.99, 0.995, 0.999]

n_tuning_episodes = 1000  # reduced episode count for faster search
tuning_results = []

for lr, gamma, decay in itertools.product(learning_rates, discount_factors, epsilon_decays):
    agent = QLearningAgent(learning_rate=lr, discount_factor=gamma, epsilon_decay=decay)
    env_tune = PricingEnv()

    for episode in range(n_tuning_episodes):
        obs, info = env_tune.reset()
        done = False
        while not done:
            action = agent.act(obs)
            next_obs, reward, terminated, truncated, info = env_tune.step(action)
            done = terminated or truncated
            agent.update(obs, action, reward, next_obs, done)
            obs = next_obs
        agent.decay_epsilon()

    # Evaluate greedily (no exploration)
    agent.epsilon = 0.0
    eval_results = run_episodes(agent, env_tune, n_episodes=100)

    tuning_results.append({
        "learning_rate": lr,
        "discount_factor": gamma,
        "epsilon_decay": decay,
        "mean_revenue": eval_results["mean_revenue"],
    })
    print(f"lr={lr}, gamma={gamma}, decay={decay} -> mean_revenue={eval_results['mean_revenue']:.2f}")

print("Tuning complete.")

## 12. Best Hyperparameter Combination

In [ ]:
best = max(tuning_results, key=lambda x: x["mean_revenue"])
print("Best hyperparameters found:")
print(f"  learning_rate   = {best['learning_rate']}")
print(f"  discount_factor = {best['discount_factor']}")
print(f"  epsilon_decay   = {best['epsilon_decay']}")
print(f"  mean_revenue    = {best['mean_revenue']:.2f}")

import pandas as pd
tuning_df = pd.DataFrame(tuning_results).sort_values("mean_revenue", ascending=False)
tuning_df.head(10)

## 13. Retrain Final Q-Learning Agent with Best Hyperparameters

In [ ]:
q_agent_final = QLearningAgent(
    learning_rate=best["learning_rate"],
    discount_factor=best["discount_factor"],
    epsilon_decay=best["epsilon_decay"],
)

env_final = PricingEnv()
final_training_rewards = []

for episode in range(5000):
    obs, info = env_final.reset()
    total_reward = 0
    done = False
    while not done:
        action = q_agent_final.act(obs)
        next_obs, reward, terminated, truncated, info = env_final.step(action)
        done = terminated or truncated
        q_agent_final.update(obs, action, reward, next_obs, done)
        obs = next_obs
        total_reward += reward
    q_agent_final.decay_epsilon()
    final_training_rewards.append(total_reward)

print("Final agent training complete.")
np.save('../outputs/trained_qtable_best.npy', q_agent_final.q_table)
print("Saved best-tuned Q-table to outputs/trained_qtable_best.npy")

## 14. Revenue Improvement — Q-Learning vs Best Heuristic Baseline

In [ ]:
q_agent_final.epsilon = 0.0
q_eval = run_episodes(q_agent_final, env_final, n_episodes=500)

heuristic_results = {
    "Random": run_episodes(RandomAgent(env_final.action_space), env_final, n_episodes=500),
    "Fixed": run_episodes(FixedPriceAgent(price_level_index=5), env_final, n_episodes=500),
    "Discount": run_episodes(TimeBasedDiscountAgent(start_price_level=9), env_final, n_episodes=500),
}

best_heuristic_name = max(heuristic_results, key=lambda k: heuristic_results[k]["mean_revenue"])
best_heuristic_revenue = heuristic_results[best_heuristic_name]["mean_revenue"]
q_revenue = q_eval["mean_revenue"]

improvement_pct = ((q_revenue - best_heuristic_revenue) / best_heuristic_revenue) * 100

print(f"Best heuristic: {best_heuristic_name} (mean revenue = {best_heuristic_revenue:.2f})")
print(f"Q-Learning mean revenue: {q_revenue:.2f}")
print(f"Revenue improvement: {improvement_pct:+.1f}%")

## 15. Ablation Summary

Q-Learning achieved a mean episodic revenue of [X], compared to [Y] for
the best-performing heuristic baseline ([baseline name]). This represents
a [Z]% improvement in revenue, driven by the agent's learned policy of
adjusting price dynamically based on remaining inventory and time to
departure — rather than following a fixed or purely time-based rule.